# Shape Boundary Tool — Prototype Test Notebook

This notebook tests the `shape_boundary_04` MCP tool end-to-end.

**What the tool does:**  
Given either a **building shape type** (`L`, `Y`, `T`, `H`, `X`, `Z`, `O`) or a **Rhino object GUID**, it retrieves the boundary polygon of the corresponding pre-defined building footprint from Rhino.

**Test plan:**
1. Load `.env` + `mcp.json` config
2. Define the tool JSON schema
3. Validate offline shape-lookup logic (no Rhino needed)
4. Connect to MCP server (Swiftlet on `localhost:3001`)
5. Discover tools and call `shape_boundary_04` live
6. Drive the tool via an LLM prompt (Cloudflare)


## 1. Setup — Imports & Path Configuration

In [9]:
import sys, os, json, math
from pathlib import Path

# ── Ensure team_04/PY is importable ──────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

# Root of the repo (AIA26_Studio)
REPO_ROOT = NOTEBOOK_DIR.parents[1]

ENV_PATH     = REPO_ROOT / ".env"
MCP_CFG_PATH = REPO_ROOT / "mcp.json"

print(f"Repo root : {REPO_ROOT}")
print(f".env      : {ENV_PATH}  (exists={ENV_PATH.exists()})")
print(f"mcp.json  : {MCP_CFG_PATH}  (exists={MCP_CFG_PATH.exists()})")


Repo root : D:\3rd sem\STUDIO\AIA26_Studio
.env      : D:\3rd sem\STUDIO\AIA26_Studio\.env  (exists=False)
mcp.json  : D:\3rd sem\STUDIO\AIA26_Studio\mcp.json  (exists=True)


## 2. Load Configuration from `.env` + `mcp.json`

In [10]:
from dotenv import load_dotenv

load_dotenv(dotenv_path=ENV_PATH, override=True)

# ── Read MCP endpoint from mcp.json ──────────────────────────────────────────
with open(MCP_CFG_PATH, encoding="utf-8") as f:
    mcp_cfg = json.load(f)

servers     = mcp_cfg["mcpServers"]
server_key  = next(iter(servers))
server_cfg  = servers[server_key]

# Endpoint is stored in args[0] (SwiftletBridge pattern)
MCP_ENDPOINT = server_cfg.get("url") or server_cfg["args"][0]

# ── Read LLM credentials (Cloudflare) ────────────────────────────────────────
CF_ACCOUNT_ID = os.environ.get("CF_ACCOUNT_ID", "")
CF_API_TOKEN  = os.environ.get("CF_API_TOKEN", "")
CF_MODEL      = os.environ.get("CF_MODEL", "@cf/meta/llama-3.3-70b-instruct-fp8-fast")

CF_BASE_URL   = f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/ai/v1"

print(f"MCP server key : {server_key}")
print(f"MCP endpoint   : {MCP_ENDPOINT}")
print(f"CF account set : {'yes' if CF_ACCOUNT_ID else 'MISSING'}")
print(f"CF token set   : {'yes' if CF_API_TOKEN else 'MISSING'}")
print(f"CF model       : {CF_MODEL}")


MCP server key : Swiftlet
MCP endpoint   : http://localhost:3001/mcp/
CF account set : MISSING
CF token set   : MISSING
CF model       : @cf/meta/llama-3.3-70b-instruct-fp8-fast


## 3. Shape Boundary Tool — JSON Schema

This is the contract the Grasshopper component must implement.  
Either `shape_type` **or** `shape_guid` is required (not both).

| Input | Type | Example | Notes |
|---|---|---|---|
| `shape_type` | string | `"L"` | Looks up pre-placed GUID from registry |
| `shape_guid` | string | `"a3b4c5d6-..."` | References any Rhino object directly |


In [11]:
SHAPE_BOUNDARY_TOOL_SCHEMA = {
    "name": "shape_boundary_04",
    "description": (
        "Retrieve the boundary polygon of a pre-defined building footprint shape from Rhino. "
        "Supply either shape_type (L/Y/T/H/X/Z/O) to look up the canonical GUID, "
        "or shape_guid to directly reference any Rhino curve object. "
        "Returns boundary coordinates, area, perimeter, centroid, and bounding box."
    ),
    "inputSchema": {
        "type": "object",
        "properties": {
            "shape_type": {
                "type": "string",
                "enum": ["L", "Y", "T", "H", "X", "Z", "O"],
                "description": "Building footprint shape code. Rhino must have a curve named/tagged with this shape pre-placed.",
            },
            "shape_guid": {
                "type": "string",
                "description": "Rhino object GUID of a curve to use directly (e.g. 'a3b4c5d6-1234-...'). Takes priority over shape_type.",
            },
        },
        # At least one of the two is required — enforced in Grasshopper logic
        "anyOf": [
            {"required": ["shape_type"]},
            {"required": ["shape_guid"]},
        ],
    },
}

print(json.dumps(SHAPE_BOUNDARY_TOOL_SCHEMA, indent=2))


{
  "name": "shape_boundary_04",
  "description": "Retrieve the boundary polygon of a pre-defined building footprint shape from Rhino. Supply either shape_type (L/Y/T/H/X/Z/O) to look up the canonical GUID, or shape_guid to directly reference any Rhino curve object. Returns boundary coordinates, area, perimeter, centroid, and bounding box.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "shape_type": {
        "type": "string",
        "enum": [
          "L",
          "Y",
          "T",
          "H",
          "X",
          "Z",
          "O"
        ],
        "description": "Building footprint shape code. Rhino must have a curve named/tagged with this shape pre-placed."
      },
      "shape_guid": {
        "type": "string",
        "description": "Rhino object GUID of a curve to use directly (e.g. 'a3b4c5d6-1234-...'). Takes priority over shape_type."
      }
    },
    "anyOf": [
      {
        "required": [
          "shape_type"
        ]
      },
    

## 4. Offline Logic Test — Shape Registry + Mock Response

Validates Python-side logic without Rhino running.  
The **shape registry** maps each shape letter to a canonical GUID.  
In Grasshopper the registry is stored as JSON on a Panel; the Python script component reads it and returns the matching boundary.


In [12]:
# ── Shape GUID registry (mirrors what lives in the Grasshopper Panel) ─────────
# Each GUID here is a placeholder that you replace with the actual Rhino object GUIDs
# after placing the footprint curves in Rhino.
SHAPE_GUID_REGISTRY: dict[str, str] = {
    "L": "00000000-0000-0000-0000-00000000000L",
    "Y": "00000000-0000-0000-0000-00000000000Y",
    "T": "00000000-0000-0000-0000-00000000000T",
    "H": "00000000-0000-0000-0000-00000000000H",
    "X": "00000000-0000-0000-0000-00000000000X",
    "Z": "00000000-0000-0000-0000-00000000000Z",
    "O": "00000000-0000-0000-0000-00000000000O",
}

# ── Mock boundary data (approximates each footprint at 1-unit grid scale) ─────
# Replace with real coordinate output from Grasshopper once curves are in Rhino.
MOCK_BOUNDARIES: dict[str, list[list[float]]] = {
    "L": [[0,0],[2,0],[2,1],[1,1],[1,3],[0,3],[0,0]],
    "Y": [[1,0],[2,1],[2,2],[1.5,2],[1.5,4],[0.5,4],[0.5,2],[0,2],[0,1],[1,0]],
    "T": [[0,0],[3,0],[3,1],[2,1],[2,3],[1,3],[1,1],[0,1],[0,0]],
    "H": [[0,0],[1,0],[1,1],[2,1],[2,0],[3,0],[3,3],[2,3],[2,2],[1,2],[1,3],[0,3],[0,0]],
    "X": [[1,0],[2,0],[2,1],[3,1],[3,2],[2,2],[2,3],[1,3],[1,2],[0,2],[0,1],[1,1],[1,0]],
    "Z": [[0,0],[3,0],[3,1],[1,1],[3,3],[0,3],[0,2],[2,2],[0,0]],
    "O": [[0,1],[1,0],[3,0],[4,1],[4,3],[3,4],[1,4],[0,3],[0,1]],
}

def _polygon_area(coords: list[list[float]]) -> float:
    """Shoelace formula — returns area of a closed polygon."""
    n = len(coords)
    area = 0.0
    for i in range(n):
        x1, y1 = coords[i]
        x2, y2 = coords[(i + 1) % n]
        area += x1 * y2 - x2 * y1
    return abs(area) / 2.0

def _polygon_perimeter(coords: list[list[float]]) -> float:
    n = len(coords)
    return sum(
        math.hypot(coords[(i+1)%n][0] - coords[i][0], coords[(i+1)%n][1] - coords[i][1])
        for i in range(n)
    )

def _centroid(coords: list[list[float]]) -> list[float]:
    xs = [p[0] for p in coords]
    ys = [p[1] for p in coords]
    return [sum(xs) / len(xs), sum(ys) / len(ys)]

def _bbox(coords: list[list[float]]) -> dict:
    xs = [p[0] for p in coords]
    ys = [p[1] for p in coords]
    return {"min": [min(xs), min(ys)], "max": [max(xs), max(ys)]}


def mock_shape_boundary(shape_type: str | None = None,
                        shape_guid: str | None = None) -> dict:
    """
    Simulates the Grasshopper shape_boundary_04 tool response.
    - shape_guid takes priority over shape_type.
    - Resolves shape_type → guid via registry, then returns mock boundary data.
    """
    if shape_guid:
        # Reverse-lookup shape type from GUID (for mock purposes)
        reverse = {v: k for k, v in SHAPE_GUID_REGISTRY.items()}
        resolved_type = reverse.get(shape_guid)
        resolved_guid = shape_guid
    elif shape_type:
        resolved_type = shape_type.upper()
        resolved_guid = SHAPE_GUID_REGISTRY.get(resolved_type)
    else:
        return {"success": False, "error": "Provide shape_type or shape_guid."}

    if resolved_type not in MOCK_BOUNDARIES:
        return {"success": False, "error": f"Unknown shape: '{resolved_type}'"}
    if not resolved_guid:
        return {"success": False, "error": f"No GUID registered for shape '{resolved_type}'"}

    coords = MOCK_BOUNDARIES[resolved_type]
    area      = _polygon_area(coords)
    perimeter = _polygon_perimeter(coords)
    centroid  = _centroid(coords)
    bbox      = _bbox(coords)

    return {
        "success": True,
        "data": {
            "shape_type":           resolved_type,
            "shape_guid":           resolved_guid,
            "boundary_coordinates": coords,
            "area_sqm":             round(area, 3),
            "perimeter_m":          round(perimeter, 3),
            "centroid":             centroid,
            "bounding_box":         bbox,
            "num_vertices":         len(coords),
        },
        "metadata": {
            "tool_name":  "shape_boundary_04",
            "mode":       "mock",
        },
    }


In [13]:
# ── Test all shape types ───────────────────────────────────────────────────────
print("=" * 60)
print("OFFLINE MOCK TESTS — All Shape Types")
print("=" * 60)

ALL_PASS = True
for shape in ["L", "Y", "T", "H", "X", "Z", "O"]:
    result = mock_shape_boundary(shape_type=shape)
    ok = result["success"]
    if ok:
        d = result["data"]
        print(f"  [{shape}]  area={d['area_sqm']:>6.2f}  perim={d['perimeter_m']:>6.2f}"
              f"  verts={d['num_vertices']:>2}  centroid={d['centroid']}  ✓")
    else:
        print(f"  [{shape}]  FAIL — {result['error']}")
        ALL_PASS = False

print()
# ── Test lookup by GUID ────────────────────────────────────────────────────────
guid_result = mock_shape_boundary(shape_guid="00000000-0000-0000-0000-00000000000L")
print("GUID lookup (L-shape GUID):")
print(f"  shape_type={guid_result['data']['shape_type']}  area={guid_result['data']['area_sqm']}  ✓")

print()
# ── Test validation: no input ──────────────────────────────────────────────────
err_result = mock_shape_boundary()
print("Validation — no input provided:")
print(f"  success={err_result['success']}  error='{err_result['error']}'  ✓" if not err_result["success"] else "  UNEXPECTED SUCCESS")

# ── Test validation: unknown shape ─────────────────────────────────────────────
bad_result = mock_shape_boundary(shape_type="Q")
print("Validation — unknown shape 'Q':")
print(f"  success={bad_result['success']}  error='{bad_result['error']}'  ✓" if not bad_result["success"] else "  UNEXPECTED SUCCESS")

print()
print("ALL OFFLINE TESTS PASSED ✓" if ALL_PASS else "SOME TESTS FAILED ✗")


OFFLINE MOCK TESTS — All Shape Types
  [L]  area=  4.00  perim= 10.00  verts= 7  centroid=[0.8571428571428571, 1.1428571428571428]  ✓
  [Y]  area=  5.00  perim= 10.83  verts=10  centroid=[1.0, 1.8]  ✓
  [T]  area=  5.00  perim= 12.00  verts= 9  centroid=[1.3333333333333333, 1.1111111111111112]  ✓
  [H]  area=  7.00  perim= 16.00  verts=13  centroid=[1.3846153846153846, 1.3846153846153846]  ✓
  [X]  area=  5.00  perim= 12.00  verts=13  centroid=[1.4615384615384615, 1.3846153846153846]  ✓
  [Z]  area=  5.00  perim= 17.66  verts= 9  centroid=[1.3333333333333333, 1.3333333333333333]  ✓
  [O]  area= 14.00  perim= 13.66  verts= 9  centroid=[1.7777777777777777, 1.8888888888888888]  ✓

GUID lookup (L-shape GUID):
  shape_type=L  area=4.0  ✓

Validation — no input provided:
  success=False  error='Provide shape_type or shape_guid.'  ✓
Validation — unknown shape 'Q':
  success=False  error='Unknown shape: 'Q''  ✓

ALL OFFLINE TESTS PASSED ✓


## 5. Unified LLM + MCP Workflow

This cell connects to both the Cloudflare LLM and the Swiftlet MCP server, runs a user prompt through the LLM with tool access, and returns the final result — just like langgraph_test.py does.

In [14]:
import httpx
import json
import time
import re
from pathlib import Path

from mcp_client import McpClient

# ──────────────────────────────────────────────────────────────────────────────
# Initialize MCP Client Connection
# ──────────────────────────────────────────────────────────────────────────────

print("=" * 70)
print("INITIALIZING MCP + LLM WORKFLOW")
print("=" * 70)

mcp: McpClient | None = None
discovered_tools: list[dict] = []

try:
    print(f"\n[1/3] Connecting to MCP server at: {MCP_ENDPOINT}")
    mcp = McpClient(endpoint=MCP_ENDPOINT, timeout_seconds=10.0)
    mcp.initialize()
    discovered_tools = mcp.list_tools()
    print(f"✓ Connected. Discovered {len(discovered_tools)} tool(s):")
    for t in discovered_tools:
        print(f"    • {t.get('name', 'unnamed')}")
except httpx.ConnectError:
    print(f"✗ Cannot reach MCP server at {MCP_ENDPOINT}")
    print("  (Rhino may not be running or Swiftlet not active)")
    mcp = None
except Exception as exc:
    print(f"✗ MCP error: {exc}")
    mcp = None

print(f"\n[2/3] LLM Configuration:")
print(f"    Provider: cloudflare")
print(f"    Model: {CF_MODEL}")
print(f"    Credentials: {'✓ Set' if CF_ACCOUNT_ID and CF_API_TOKEN else '✗ Missing'}")

print(f"\n[3/3] Tool Schema for LLM:")
print(f"    Target tool: shape_boundary_04")
print(f"    Schema size: {len(json.dumps(SHAPE_BOUNDARY_TOOL_SCHEMA))} bytes")

INITIALIZING MCP + LLM WORKFLOW

[1/3] Connecting to MCP server at: http://localhost:3001/mcp/
✓ Connected. Discovered 3 tool(s):
    • site_boundary_generation
    • Usable_site_area
    • parametric_shape_generator

[2/3] LLM Configuration:
    Provider: cloudflare
    Model: @cf/meta/llama-3.3-70b-instruct-fp8-fast
    Credentials: ✗ Missing

[3/3] Tool Schema for LLM:
    Target tool: shape_boundary_04
    Schema size: 810 bytes


## 6. Live MCP Tool Call — `shape_boundary_04`

Call the tool with **shape_type** (text input) and then again with **shape_guid** (GUID input).  
If Rhino is not running, the call will raise a connection error — that is expected.


In [15]:
# ──────────────────────────────────────────────────────────────────────────────
# LLM + Tool Integration Functions
# ──────────────────────────────────────────────────────────────────────────────

TOOL_NAME = "shape_boundary_04"

def call_llm_for_tool(user_message: str) -> str:
    """Call Cloudflare LLM with tool schema in system prompt."""
    
    system_prompt = f"""You are a building design AI assistant.

You have access to ONE tool:

{json.dumps(SHAPE_BOUNDARY_TOOL_SCHEMA, indent=2)}

RULES:
- When the user asks about a building shape boundary, call shape_boundary_04.
- shape_type must be one of: L, Y, T, H, X, Z, O.
- If the user provides a Rhino GUID, use shape_guid instead.
- Respond ONLY with a JSON tool-call object — no prose, no markdown fences.

Example response:
{{
  "action": "tool",
  "tool_calls": [
    {{
      "name": "shape_boundary_04",
      "arguments": {{
        "shape_type": "L"
      }}
    }}
  ]
}}
"""
    
    url = f"{CF_BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {CF_API_TOKEN}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": CF_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
        "max_tokens": 300,
        "temperature": 0.0,
    }
    
    try:
        response = httpx.post(url, json=payload, headers=headers, timeout=30.0)
        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"]
        if isinstance(content, dict):
            return json.dumps(content)
        return str(content)
    except httpx.HTTPStatusError as e:
        return f"{{\"error\": \"HTTP {e.response.status_code}: {e.response.text[:200]}\"}}"
    except Exception as e:
        return f"{{\"error\": \"{str(e)}\"}}"


def extract_tool_call(llm_response: str) -> dict | None:
    """Parse the LLM's JSON response and extract the tool call."""
    cleaned = re.sub(r"```(?:json)?|```", "", llm_response).strip()
    try:
        parsed = json.loads(cleaned)
        calls = parsed.get("tool_calls", [])
        return calls[0] if calls else None
    except json.JSONDecodeError:
        return None


def execute_tool(tool_name: str, tool_args: dict) -> dict:
    """Execute tool via MCP, with fallback to mock."""
    if tool_name != TOOL_NAME:
        return {"success": False, "error": f"Unknown tool '{tool_name}'"}
    
    if mcp is not None:
        registered = {t.get("name") for t in discovered_tools}
        if TOOL_NAME in registered:
            try:
                raw = mcp.call_tool(TOOL_NAME, tool_args)
                return json.loads(raw) if raw.strip().startswith("{") else {"success": True, "raw": raw}
            except Exception as exc:
                print(f"[MCP Error] {exc} — falling back to mock")
    
    # Fall back to mock
    return mock_shape_boundary(
        shape_type=tool_args.get("shape_type"),
        shape_guid=tool_args.get("shape_guid")
    )

In [16]:
# ──────────────────────────────────────────────────────────────────────────────
# Execute Workflow: User Prompt → LLM → Tool Call → Result
# ──────────────────────────────────────────────────────────────────────────────

TEST_PROMPTS = [
    "What is the boundary of an L-shaped building?",
    "Show me the footprint of a T-shaped building.",
]

for idx, user_prompt in enumerate(TEST_PROMPTS, 1):
    print(f"\n{'=' * 70}")
    print(f"TEST {idx}: {user_prompt}")
    print("=" * 70)
    
    # Step 1: Call LLM
    print("\n→ Calling Cloudflare LLM...")
    llm_response = call_llm_for_tool(user_prompt)
    print(f"LLM response (first 300 chars):\n{llm_response[:300]}")
    
    # Step 2: Extract tool call
    print("\n→ Parsing tool call...")
    tool_call = extract_tool_call(llm_response)
    if not tool_call:
        print("✗ LLM did not return a recognizable tool call")
        continue
    
    tool_name = tool_call.get("name")
    tool_args = tool_call.get("arguments", {})
    print(f"✓ Extracted: {tool_name}({tool_args})")
    
    # Step 3: Execute tool
    print("\n→ Executing tool...")
    start = time.perf_counter()
    result = execute_tool(tool_name, tool_args)
    elapsed = (time.perf_counter() - start) * 1000
    
    # Step 4: Display result
    print(f"✓ Result (in {elapsed:.0f}ms):")
    if result.get("success"):
        data = result.get("data", {})
        print(f"  Shape: {data.get('shape_type')}")
        print(f"  Area: {data.get('area_sqm')} sqm")
        print(f"  Perimeter: {data.get('perimeter_m')} m")
        print(f"  Vertices: {data.get('num_vertices')}")
        print(f"  Centroid: {data.get('centroid')}")
    else:
        print(f"  Error: {result.get('error')}")
    
    print()


TEST 1: What is the boundary of an L-shaped building?

→ Calling Cloudflare LLM...
LLM response (first 300 chars):
{"error": "Illegal header value b'Bearer '"}

→ Parsing tool call...
✗ LLM did not return a recognizable tool call

TEST 2: Show me the footprint of a T-shaped building.

→ Calling Cloudflare LLM...
LLM response (first 300 chars):
{"error": "Illegal header value b'Bearer '"}

→ Parsing tool call...
✗ LLM did not return a recognizable tool call


## 7. LLM Integration — Cloudflare + Tool-Call Parsing

Sends a natural-language prompt to the Cloudflare LLM.  
The LLM is given the `shape_boundary_04` schema in its system prompt and must respond with a JSON tool-call.  
We then execute it and return the result.


In [17]:
# ──────────────────────────────────────────────────────────────────────────────
# Debug: Check State After Workflow
# ──────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("DEBUG OUTPUT")
print("=" * 70)
print(f"\nMCP connection: {mcp is not None}")
print(f"Discovered tools: {len(discovered_tools)}")
print(f"Tool name: {TOOL_NAME}")
print(f"Last LLM response (first 200 chars):\n{llm_response[:200] if 'llm_response' in dir() else 'N/A'}")
print(f"Tool call extracted: {tool_call is not None}")
print(f"Tool call value: {tool_call}")



DEBUG OUTPUT

MCP connection: True
Discovered tools: 3
Tool name: shape_boundary_04
Last LLM response (first 200 chars):
{"error": "Illegal header value b'Bearer '"}
Tool call extracted: False
Tool call value: None


## 6. Summary

This notebook demonstrates a **full LLM + MCP integration**:

1. **Offline validation** — Shape geometry logic works standalone
2. **MCP connection** — Connects to Swiftlet on localhost:3001
3. **LLM integration** — Sends user prompts to Cloudflare with tool schemas
4. **Tool execution** — Calls `shape_boundary_04` via MCP or mock
5. **Result parsing** — Extracts and displays boundary data

### Expected Behavior

- **All sections work** if `.env` has valid Cloudflare credentials
- **Sections 5-6 work best** with Rhino + Swiftlet running
- **Mock fallback** ensures workflow runs even without Rhino (using synthetic data)
- **Similar to** `langgraph_test.ipynb` — unified LLM + MCP workflow